# workflows

> Fill in a module description here

## Agentic Workflow Design for Linked Data Navigation

1. **Initial Exploration Workflow**:
   - Start with a high-level exploration of the dataset structure
   - Identify key vocabularies and schemas being used
   - Locate important structures like RecordSet, Fields, etc.

2. **Vocabulary Understanding Workflow**:
   - When the agent encounters an unfamiliar term, it should look up its definition
   - Follow relationships to understand how concepts relate to each other
   - Build a mental model of the vocabulary structure

3. **Data Structure Analysis Workflow**:
   - Navigate from the dataset root to key structures like RecordSet
   - Analyze the fields and their data types
   - Trace connections between data elements and their definitions

4. **Question Answering Workflow**:
   - Gather evidence from both the vocabulary and dataset
   - Combine understanding of terms with actual data structure
   - Formulate comprehensive answers with citations

The key is that the LLM should be able to chain these workflows together as needed, using the existing tools we've built:

- `find_entity` to locate definitions in vocabularies
- `get_entity_description` to understand those definitions
- `explore_dataset` to navigate through the actual dataset structure
- `search_dataset` to find specific elements

With these tools, the LLM can perform tasks like:
1. "What is a RecordSet according to the Croissant vocabulary?"
2. "How are Fields defined in this dataset?"
3. "What are the data types of each field in the Glass dataset?"
4. "How does this dataset implement the Croissant structure?"

The agent can navigate between vocabulary definitions and dataset implementations without needing additional code. It can follow the links between concepts, understand their definitions, and see how they're applied in practice.

This approach leverages the power of linked data - the ability to navigate connections between entities - while letting the LLM handle the reasoning and synthesis aspects through carefully designed prompt chains.

In [ ]:
#| default_exp workflows

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| exports

from fastcore.basics import *
from fastcore.meta import *
from fastcore.test import *
from IPython.display import Markdown, display
from cogitarelink.core import LinkedDataKnowledge
from cogitarelink.navigation import *
from claudette import Chat, tool, toolloop, models
from cogitarelink.tools import find_vocabulary_term, follow_relationship, explore_dataset, search_dataset, get_dataset_evidence
import json
from typing import List, Dict, Any, Optional, Union, Set

In [ ]:
model = models[2]
model

'claude-3-5-sonnet-20241022'

In [ ]:
#| export
def create_workflow(
    steps:List[str], # List of step instructions
    summary:str="Summarize what you've learned from this exploration." # Summary instruction
) -> str: # Workflow prompt
    "Create a simple step-by-step workflow prompt from a list of instructions"
    prompt = "I want you to explore this data step by step:\n\n"
    
    for i, step in enumerate(steps, 1):
        prompt += f"Step {i}: {step}\n\n"
    
    prompt += f"""For each step, use the appropriate tools to gather evidence, and explain what you find before moving to the next step.

{summary}"""
    
    return prompt

In [ ]:
#| test
def test_create_workflow():
    steps = [
        "Explore the definition of Person in the vocabulary.",
        "Find what relationships Person has with other concepts.",
        "Examine how Person is implemented in the dataset."
    ]
    
    prompt = create_workflow(steps)
    
    assert("Step 1: Explore the definition of Person" in prompt)
    assert("Step 2: Find what relationships" in prompt)
    assert("Step 3: Examine how Person" in prompt)
    assert("For each step, use the appropriate tools" in prompt)
    assert("Summarize what you've learned" in prompt)
    
    # Test with custom summary
    custom_summary = "Create a comprehensive report on Person."
    prompt = create_workflow(steps, custom_summary)
    assert(custom_summary in prompt)


In [ ]:
test_create_workflow()

In [ ]:
#| export
def run_workflow(
    agent, # Claude agent with tools
    workflow:str, # Workflow prompt
    kb=None, # Optional vocabulary knowledge base
    dataset_kb=None, # Optional dataset knowledge base
    trace=False # Whether to print the interaction trace
) -> Any: # Agent response
    "Run a workflow with the given agent and knowledge bases"
    # Set global variables if provided
    if kb is not None: globals()['kb'] = kb
    if dataset_kb is not None: globals()['dataset_kb'] = dataset_kb
    
    # Run the workflow
    trace_func = print if trace else None
    return agent.toolloop(workflow, trace_func=trace_func)

In [ ]:
def test_run_workflow():
    # Create a mock agent
    class MockAgent:
        def toolloop(self, prompt, trace_func=None):
            # Just return the prompt for testing
            return prompt
    
    mock_agent = MockAgent()
    
    # Create a simple workflow
    workflow = create_workflow(["Test step"])
    
    # Run the workflow with the mock agent
    result = run_workflow(mock_agent, workflow)
    
    # Check that the result is the workflow prompt
    test_eq(result, workflow)
    
    # Test with knowledge bases
    kb = LinkedDataKnowledge({"@graph": []})
    dataset_kb = LinkedDataKnowledge({"data": []})
    
    # Save the original global values if they exist
    orig_kb = globals().get('kb', None)
    orig_dataset_kb = globals().get('dataset_kb', None)
    
    try:
        # Run with knowledge bases
        run_workflow(mock_agent, workflow, kb, dataset_kb)
        
        # Check that the global variables were set
        test_is(globals().get('kb'), kb)
        test_is(globals().get('dataset_kb'), dataset_kb)
    finally:
        # Restore original values
        if orig_kb is not None:
            globals()['kb'] = orig_kb
        else:
            globals().pop('kb', None)
            
        if orig_dataset_kb is not None:
            globals()['dataset_kb'] = orig_dataset_kb
        else:
            globals().pop('dataset_kb', None)

In [ ]:
#| export
def create_agent(
    model:str="claude-3-5-sonnet-20241022", # Claude model to use
    sp:str=None # Optional system prompt
) -> Chat: # Chat agent
    "Create a Claude agent with linked data tools"
    if sp is None:
        sp = """You are a Linked Data Navigator that helps users explore and understand structured data.
When exploring linked data:
- Always cite specific paths and values from the data
- Follow relationships between concepts to build understanding
- Explain both the vocabulary definitions and how they're implemented in actual data
- Use a step-by-step approach to build comprehensive understanding"""
    
    return Chat(
        model=model,
        sp=sp,
        tools=[
            find_vocabulary_term,
            follow_relationship,
            explore_dataset,
            search_dataset,
            get_dataset_evidence
        ]
    )

In [ ]:
#| test
def test_create_agent():
    # Mock the Chat class
    original_chat = Chat
    
    try:
        # Create a mock Chat class that we can inspect
        class MockChat:
            def __init__(self, model, sp, tools):
                self.model = model
                self.sp = sp
                self.tools = tools
        
        # Replace the real Chat with our mock
        globals()['Chat'] = MockChat
        
        # Test creating an agent
        agent = create_agent()
        
        # Check that the agent has the expected configuration
        test_eq(agent.model, "claude-3-5-sonnet-20241022")
        assert("Linked Data Navigator" in agent.sp)
        test_eq(len(agent.tools), 5)
        
        # Test with custom model and system prompt
        custom_sp = "You are a data explorer."
        agent = create_agent(model="claude-3-opus-20240229", sp=custom_sp)
        test_eq(agent.model, "claude-3-opus-20240229")
        test_eq(agent.sp, custom_sp)
        
    finally:
        # Restore the original Chat class
        globals()['Chat'] = original_chat

In [ ]:
test_create_agent()

In [ ]:
test_run_workflow()

In [ ]:
#| eval
def example_usage():
    """Example of how to use the workflows module.
    
    This is just an example and is not executed automatically.
    """
    # Create knowledge bases
    kb = LinkedDataKnowledge({
        "@context": {},
        "@graph": [
            {
                "@id": "http://example.org/Person",
                "@type": "rdfs:Class",
                "rdfs:label": "Person",
                "rdfs:comment": "A person (alive, dead, undead, or fictional)."
            }
        ]
    })
    
    dataset_kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "people": [
            {
                "@id": "person1",
                "@type": "Person",
                "name": "Alice",
                "email": "alice@example.com"
            }
        ]
    })
    
    # Create an agent
    agent = create_agent()
    
    # Create a vocabulary exploration workflow
    workflow = vocab_exploration_workflow("Person")
    
    # Run the workflow
    result = run_workflow(agent, workflow, kb, dataset_kb, trace=True)
    
    # Print the result
    print(result)
    
    return result


In [ ]:
example_usage()

NameError: name 'vocab_exploration_workflow' is not defined

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()